In [1]:
# Our markdown document collection

import pathlib

compass_documents_path = pathlib.Path("compass_documents")

compass_documents = list(compass_documents_path.glob("*.md"))

compass_documents[0]

WindowsPath('compass_documents/annual_leave_planning.md')

In [2]:
# Let's sample some chunking
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)


def make_to_text(md_header_split: str) -> str:
    chunk = ""
    some_title = md_header_split.metadata.get("Header 1")
    some_section = md_header_split.metadata.get("Header 2")
    some_subsection = md_header_split.metadata.get("Header 3")

    if some_title:
        chunk = f"This text talks about {some_title}"
    if some_section:
        chunk = f"{chunk}. It is specific on {some_section}"
    if some_subsection:
        chunk = f"{chunk}. and {some_section}."
    else:
        chunk = f"{chunk}."

    return f"{chunk} Details follow {md_header_split.page_content}"


# Chunk a Markdown document
with compass_documents[0].open("r", encoding="utf-8") as f:
    markdown_document = f.read()

md_header_splits = markdown_splitter.split_text(markdown_document)


for md_header_split in md_header_splits:

    chunk = make_to_text(md_header_split)

    print(chunk)

c:\work\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\work\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


. Details follow ---
title: Annual Leave Planning
category: HR
topic: annual_leave_planning
language: en
source: internal_policy
doc_type: guide
---
This text talks about Annual Leave Planning. Details follow As annual leave must be administered by the end of March of the following calendar year, it is very important to plan vacation days as early as possible in order to avoid conflict with your workflow.
This text talks about Annual Leave Planning. It is specific on Planning Process. Details follow - Annual Leave Planning is carried out by submitting leave and maintaining it in a "draft" status.
- "Draft" leave is editable in order to make any changes required. When the dates have been finalised (always upon agreement with your team and/or customer), the standard leave request procedure must be followed.
- Once the relevant absence form has been sent to `absences@agileactors.com`, your annual leave submission status will be changed to "Approved".
This text talks about Annual Leave Pla

In [3]:
import weaviate
from weaviate.classes.config import Configure

# Step 1.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:
    # Step 1.2: Create a collection
    client.collections.delete(name="vectordb_tutorial")
    vectordb_tutorial = client.collections.create(
        name="vectordb_tutorial",
        vector_config=Configure.Vectors.text2vec_ollama(  # Configure the Ollama embedding integration
            api_endpoint="http://ollama:11434",  # If using Docker you might need: http://host.docker.internal:11434
            model="nomic-embed-text",  # The model to use
        ),
    )

In [4]:
documents = []
for compass_document in compass_documents:
    with compass_document.open("r", encoding="utf-8") as f:
        markdown_document = f.read()
        md_header_splits = markdown_splitter.split_text(markdown_document)
        print(f"Processing document: {compass_document.name}")
        for md_header_split in md_header_splits:
            document_chunk = make_to_text(md_header_split)
            document = {
                "name_of_document": compass_document.name,
                "document_chunk": document_chunk,
            }
            documents.append(document)

Processing document: annual_leave_planning.md
Processing document: business-travel-policy.md
Processing document: chapters.md
Processing document: Emergency_Response_Plan.md
Processing document: family_support_leave_categories.md
Processing document: GDPR.md
Processing document: leave-categories.md
Processing document: leave-guide.md
Processing document: leaves-greek.md
Processing document: MetLife.md
Processing document: offboarding.md
Processing document: oncall_callout.md
Processing document: pending_military_status.md
Processing document: pension-plan.md
Processing document: referral.md
Processing document: ticket_compliment.md
Processing document: ticket_restaurant.md
Processing document: timesheet_process.md
Processing document: work_premises_and_hours.md


In [5]:
import tqdm

with weaviate.connect_to_local() as client:
    vectordb_tutorial = client.collections.use("vectordb_tutorial")
    with vectordb_tutorial.batch.dynamic() as batch:
        for i in tqdm.tqdm(range(len(documents))):
            batch.add_object(documents[i])


print("Imported & vectorized documentss into the vectordb_tutorial collection")

100%|██████████| 208/208 [00:06<00:00, 34.58it/s]


Imported & vectorized documentss into the vectordb_tutorial collection


In [6]:
from weaviate.classes.generate import GenerativeConfig

# Step 2.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 2.2: Use this collection
    vectordb_tutorial = client.collections.use("vectordb_tutorial")

    # Step 2.3: Perform RAG with on NearText results
    response = vectordb_tutorial.generate.near_text(
        query="Paid leaves",
        limit=1,
        single_prompt="Please give a list of paid leave categories",
        grouped_task="Summarize the information",
        generative_provider=GenerativeConfig.ollama(  # Configure the Ollama generative integration
            api_endpoint="http://ollama:11434",  # If using Docker you might need: http://host.docker.internal:11434
            model="sam860/lfm2.5:1.2b",  # The model to use
        ),
    )

    print(response.generative.text)  # Inspect the results

The document provides guidance on navigating a leave policy, outlining the steps and procedures for employees regarding time off and related processes. It serves as a reference for understanding how to manage leaves according to organizational guidelines.
